### renomear tabelas bronze e silver

In [0]:
database_name_old = "workspace.default"
database_name_new = "workspace.bronze"
table_name_old = "inflacao_raw"
table_name_new = "inflacao"

spark.sql(
    f"""ALTER TABLE {database_name_old}.{table_name_old} 
    RENAME TO {database_name_new}.{table_name_new};
"""    
)

In [0]:
database_name_old = "workspace.default"
database_name_new = "workspace.silver"
table_name_old = "inflacao_cleaned"
table_name_new = "inflacao"

spark.sql(
    f"""ALTER TABLE {database_name_old}.{table_name_old} 
    RENAME TO {database_name_new}.{table_name_new};
"""    
)

### fazer tabela gold

In [0]:
from pyspark.sql import functions as F

silver_database = 'workspace.silver'
silver_table = 'inflacao'

# Agregação por Moeda

spark.table(f"{silver_database}.{silver_table}")

df_final = spark.sql("""
SELECT 
    id,
    periodo_moeda,
    variacao_mensal
FROM silver_inflacao
""")
gold_moedas_resumo = df_final.groupBy("periodo_moeda").agg(
    F.round(F.mean("variacao_mensal"), 2).alias("inflacao_mensal_media"),
    F.round(F.min("variacao_mensal"), 2).alias("inflacao_mensal_minima"),
    F.round(F.max("variacao_mensal"), 2).alias("inflacao_mensal_maxima"),
    F.round(F.stddev("variacao_mensal"), 2).alias("volatilidade_variacao"), # Mede a instabilidade
    F.count("id").alias("total_meses_duracao")
).orderBy(F.desc("inflacao_mensal_media"))

display(gold_moedas_resumo)


In [0]:
gold_database = "workspace.gold"
gold_table = "inflacao"

(
    gold_moedas_resumo.write.format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable(f"{gold_database}.{gold_table}")
)

## Funcionalidades Delta 

In [0]:
%sql
-- histórico

describe history workspace.gold.inflacao

In [0]:
%sql
-- time travel

-- Versão atual 
-- select * from workspace.gold.inflacao version as of 1 limit 5 

-- 'Rollback' para versão anterior
select * from workspace.gold.inflacao version as of 0 limit 5;

In [0]:
%sql
-- governança

describe detail workspace.gold.inflacao;